# Forward Propagation

> Based on Stanford CS230 — C1M3, Andrew Ng / DeepLearning.AI

Forward propagation is the process of passing inputs through a neural network layer-by-layer to compute the prediction $\hat{y}$.

## Learning Objectives

1. Write the general vectorised forward propagation equations for an $L$-layer network
2. Understand the shapes of weight matrices, activation matrices, and bias vectors
3. Implement forward propagation for a 2-layer network from scratch
4. Distinguish between hyperparameters and learnable parameters

## Network Notation

For layer $l$ in a network with $L$ total layers:

| Symbol | Meaning | Shape |
|---|---|---|
| $n^{[l]}$ | Number of units in layer $l$ | scalar |
| $W^{[l]}$ | Weight matrix for layer $l$ | $(n^{[l]}, n^{[l-1]})$ |
| $b^{[l]}$ | Bias vector for layer $l$ | $(n^{[l]}, 1)$ |
| $Z^{[l]}$ | Pre-activation (linear) | $(n^{[l]}, m)$ |
| $A^{[l]}$ | Post-activation output | $(n^{[l]}, m)$ |
| $g^{[l]}$ | Activation function for layer $l$ | — |

$A^{[0]} = X$ is the input, $\hat{Y} = A^{[L]}$ is the output.


## The Forward Pass Equations

For each layer $l = 1, 2, \ldots, L$:

$$Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$$
$$A^{[l]} = g^{[l]}\!\left(Z^{[l]}\right)$$

The bias $b^{[l]}$ is broadcast across all $m$ columns.

## Data Flow (2-layer network, $m$ examples)

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 160" width="640" height="160" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- boxes -->
  <rect x="20"  y="50" width="90" height="60" rx="6" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.8"/>
  <rect x="160" y="50" width="100" height="60" rx="6" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.8"/>
  <rect x="320" y="50" width="100" height="60" rx="6" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.8"/>
  <rect x="480" y="50" width="80" height="60" rx="6" fill="#e05c5c" fill-opacity="0.15" stroke="#e05c5c" stroke-width="1.8"/>
  <!-- labels inside boxes -->
  <text x="65"  y="77"  text-anchor="middle" fill="#3a7bbf" font-size="12" font-weight="bold">A&#x2070;</text>
  <text x="65"  y="93"  text-anchor="middle" fill="#3a7bbf" font-size="10">= X</text>
  <text x="65"  y="107" text-anchor="middle" fill="#3a7bbf" font-size="10">(n&#x2080;, m)</text>
  <text x="210" y="74"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Z&#x00B9; = W&#x00B9;A&#x2070;+b&#x00B9;</text>
  <text x="210" y="90"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">A&#x00B9; = g&#x00B9;(Z&#x00B9;)</text>
  <text x="210" y="106" text-anchor="middle" fill="#3d7a5e" font-size="10">(n&#x00B9;, m)</text>
  <text x="370" y="74"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Z&#x00B2; = W&#x00B2;A&#x00B9;+b&#x00B2;</text>
  <text x="370" y="90"  text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">A&#x00B2; = g&#x00B2;(Z&#x00B2;)</text>
  <text x="370" y="106" text-anchor="middle" fill="#3d7a5e" font-size="10">(n&#x00B2;, m)</text>
  <text x="520" y="74"  text-anchor="middle" fill="#b03a3a" font-size="12" font-weight="bold">&#x177; = A&#x00B2;</text>
  <text x="520" y="93"  text-anchor="middle" fill="#b03a3a" font-size="10">sigmoid</text>
  <text x="520" y="107" text-anchor="middle" fill="#b03a3a" font-size="10">(1, m)</text>
  <!-- arrows -->
  <line x1="110" y1="80" x2="157" y2="80" stroke="#888" stroke-width="1.5"/>
  <polygon points="158,80 150,76 150,84" fill="#888"/>
  <line x1="260" y1="80" x2="317" y2="80" stroke="#888" stroke-width="1.5"/>
  <polygon points="318,80 310,76 310,84" fill="#888"/>
  <line x1="420" y1="80" x2="477" y2="80" stroke="#888" stroke-width="1.5"/>
  <polygon points="478,80 470,76 470,84" fill="#888"/>
  <!-- weight labels above arrows -->
  <text x="134" y="70" text-anchor="middle" fill="#888" font-size="10" font-style="italic">W&#x00B9;, b&#x00B9;</text>
  <text x="290" y="70" text-anchor="middle" fill="#888" font-size="10" font-style="italic">W&#x00B2;, b&#x00B2;</text>
  <!-- title -->
  <text x="320" y="148" text-anchor="middle" fill="#666" font-size="11">2-layer network: n&#x2080; inputs → n&#x00B9; hidden → 1 output</text>
</svg>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# -------------------------------------------------------------------
# Forward propagation from scratch (2-layer network)
# -------------------------------------------------------------------
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

def init_params(layer_dims, seed=1):
    """Xavier initialisation."""
    np.random.seed(seed)
    params = {}
    for l in range(1, len(layer_dims)):
        scale = np.sqrt(1 / layer_dims[l - 1])
        params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * scale
        params[f'b{l}'] = np.zeros((layer_dims[l], 1))
    return params

def forward_prop(X, params, L):
    """
    Vectorised forward pass.
    Returns (AL, cache) where cache = [(A_prev, W, b, Z), ...] for each layer.
    """
    cache = []
    A = X
    for l in range(1, L + 1):
        W = params[f'W{l}']
        b = params[f'b{l}']
        Z = W @ A + b
        A_prev = A
        A = relu(Z) if l < L else sigmoid(Z)
        cache.append((A_prev, W, b, Z))
    return A, cache

# ---- Example: 4-layer network on random data ----
np.random.seed(42)
n0, m = 10, 200
X = np.random.randn(n0, m)

layer_dims = [n0, 8, 5, 3, 1]
params = init_params(layer_dims)
AL, cache = forward_prop(X, params, L=len(layer_dims) - 1)

print("Layer shapes during forward pass:")
print(f"  A[0] = X:   {X.shape}")
for l, (A_prev, W, b, Z) in enumerate(cache, start=1):
    A = relu(Z) if l < len(cache) else sigmoid(Z)
    print(f"  Layer {l}: W{l}{W.shape}  Z{l}{Z.shape}  A{l}{A.shape}")
print(f"\nOutput ŷ: {AL.shape}  — values in [{AL.min():.3f}, {AL.max():.3f}]")

# ---- Visualise activation magnitudes across layers ----
fig, axes = plt.subplots(1, len(layer_dims) - 1, figsize=(14, 4))
fig.suptitle('Activation Distributions by Layer (ReLU hidden, sigmoid output)', fontsize=13)

for l, (A_prev, W, b, Z) in enumerate(cache, start=1):
    ax = axes[l - 1]
    A = relu(Z) if l < len(cache) else sigmoid(Z)
    ax.hist(A.flatten(), bins=30, color=P[l % len(P)], alpha=0.85, edgecolor='white')
    ax.set_title(f'Layer {l}  ({A.shape[0]} units)', fontsize=10)
    ax.set_xlabel('Activation value', fontsize=9)
    ax.set_ylabel('Count' if l == 1 else '', fontsize=9)
    ax.grid(True)

plt.tight_layout()
plt.show()


## Shape Cheat Sheet

For a layer with $n^{[l]}$ units, $n^{[l-1]}$ inputs, and $m$ examples:

| Quantity | Shape | Notes |
|---|---|---|
| $W^{[l]}$ | $(n^{[l]},\; n^{[l-1]})$ | One row per unit in layer $l$ |
| $b^{[l]}$ | $(n^{[l]},\; 1)$ | Broadcast across all $m$ examples |
| $Z^{[l]}$ | $(n^{[l]},\; m)$ | One column per training example |
| $A^{[l]}$ | $(n^{[l]},\; m)$ | Same shape as $Z^{[l]}$ |

**Memory cost** grows with $m$ (mini-batch) not with $n^{[l-1]}$ — vectorisation is free in the data dimension.

> **Common bug:** transposing $W$ vs not — always verify `.shape` after computing $Z = W A + b$.
